In [ ]:
import pandas as pd
import numpy as np
from google.colab import files
files.upload()

In [ ]:
df=pd.read_csv("Community Healthcare MultiSymptomsDisease Diagnostic Dataset.csv")
df.head()

,disease,symptom_cough,symptom_dry_cough,symptom_productive_cough,symptom_shortness_of_breath,symptom_wheezing,symptom_chest_tightness,symptom_sore_throat,symptom_runny_nose,symptom_nasal_congestion,...,rare_symptom_tall_stature_rare,rare_symptom_hyperextensible_skin_rare,rare_symptom_joint_hypermobility_rare,rare_symptom_macroglossia_rare,rare_symptom_hearing_loss_early_onset_rare,rare_symptom_developmental_delay_rare,rare_symptom_proximal_muscle_weakness_rare,rare_symptom_bulbar_symptoms_rare,rare_symptom_respiratory_failure_rare,rare_symptom_renal_cysts_rare
0,Hyperthyroidism,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Coronary_Artery_Disease,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Irritable_Bowel_Syndrome,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Irritable_Bowel_Syndrome,0,0,0,0,0,1,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,Obesity,0,0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df.isnull().sum()

,0
disease,0
symptom_cough,0
symptom_dry_cough,0
symptom_productive_cough,0
symptom_shortness_of_breath,0
...,...
rare_symptom_developmental_delay_rare,0
rare_symptom_proximal_muscle_weakness_rare,0
rare_symptom_bulbar_symptoms_rare,0
rare_symptom_respiratory_failure_rare,0


In [112]:
df['disease'].unique()

array(['Hyperthyroidism', 'Coronary_Artery_Disease',
       'Irritable_Bowel_Syndrome', 'Obesity', 'Asthma', 'Osteoarthritis',
       'Pneumonia', 'Chronic_Kidney_Disease', 'Polycystic_Ovary_Syndrome',
       'Gastritis', 'Flu', 'Peptic_Ulcer_Disease', 'Hypothyroidism',
       'Type2_Diabetes', 'Common_Cold', 'Hypertension', 'COVID19',
       'Depression', 'Migraine', 'COPD', 'Ehlers_Danlos_Syndrome',
       'Cystic_Fibrosis', 'Epilepsy', 'Generalized_Anxiety_Disorder',
       'Systemic_Lupus_Erythematosus', 'Stroke', 'Osteopetrosis',
       'Amyotrophic_Lateral_Sclerosis', 'Inflammatory_Bowel_Disease',
       'Tuberculosis', 'Marfan_Syndrome', 'Spinal_Muscular_Atrophy',
       'Chronic_Liver_Disease', 'Cirrhosis', 'Gaucher_Disease',
       'Niemann_Pick_Disease', 'Dengue', 'Ulcerative_Colitis',
       'Typhoid_Fever', 'Multiple_Sclerosis', 'Crohn_Disease',
       'Acute_Gastroenteritis', 'Fanconi_Anemia', 'Psoriasis',
       'Urinary_Tract_Infection'], dtype=object)

In [ ]:
df.shape

(100000, 175)

In [ ]:
counts = df["disease"].value_counts()

valid_diseases = counts[counts > 1].index

df = df[df["disease"].isin(valid_diseases)]

In [ ]:
df.shape

(99999, 175)

In [ ]:
X = df.drop("disease", axis=1)
y = df["disease"]

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    eval_metric="mlogloss",
    use_label_encoder=False
)

model.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [13:13:10] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, ...)

In [ ]:
y_pred = model.predict(X_test)

# convert back to disease names
y_pred_labels = le.inverse_transform(y_pred)
y_test_labels = le.inverse_transform(y_test)

In [ ]:
from sklearn.metrics import accuracy_score

print("Accuracy:", accuracy_score(y_test_labels, y_pred_labels))

Accuracy: 0.66645


In [ ]:
import numpy as np

probs = model.predict_proba(X_test)
top3 = np.argsort(probs, axis=1)[:, -3:]

correct = 0
for i in range(len(y_test)):
    if y_test[i] in top3[i]:
        correct += 1

print("Top-3 Accuracy:", correct / len(y_test))

Top-3 Accuracy: 0.90485


In [ ]:
import joblib

joblib.dump(model, "disease_model.pkl")
joblib.dump(le, "label_encoder.pkl")
joblib.dump(list(X.columns), "symptoms.pkl")

print("Saved successfully")

Saved successfully


In [ ]:
files.download